# 02 · MedSigLIP vs CLIP vs CNN embeddings for chest X-ray view synthesis

> **Research use only.** Generated radiographs and CT volumes are synthetic and patient-conditioned. They are
> estimates, not acquisitions of the patient and not clinical ground truth. Never use them for diagnosis or treatment.
> Upload only images you are authorised to use, and remove burned-in identifiers first.

**Question:** which visual representation carries information that a view-synthesis model could use, and could
it replace what Parallax (SV-DRR) conditions on today?

**Compared:** `google/medsiglip-448` (gated), Parallax's own CLIP image encoder (its real conditioning signal), and the
TorchXRayVision DenseNet-121 (a conventional CXR CNN).

**Probes:** zero-shot PA/AP/lateral. Cosine similarity of the PA vs its mirror image, its upside-down copy, the same
subject's lateral, and noise. An embedding that cannot tell a PA from its mirror image carries little laterality
or geometry information. With one public subject these probes are *descriptive only*.

In [ ]:
#@title 1. Configuration { display-mode: "form" }
REPO_URL = "https://github.com/sanafathima-00/RadioVue.git"  #@param {type:"string"}
REPO_BRANCH = "claude/single-xray-vlm-pipeline"  #@param {type:"string"}
#@markdown If the pipeline code is not pushed yet, upload `radiovue_colab_bundle.zip` (see SETUP_GUIDE.md) to this Drive path:
CODE_BUNDLE_ON_DRIVE = "RadioVue/code/radiovue_colab_bundle.zip"  #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/RadioVue"  #@param {type:"string"}
DVG_CHECKPOINTS_ON_DRIVE = DRIVE_ROOT + "/Volumea/models/DVG-Diffusion"
PARALLAX_MODEL_ON_DRIVE = DRIVE_ROOT + "/Parallax/models/DiT-fb-512"  # used if present, else downloaded from Hugging Face
OUTPUT_ROOT = DRIVE_ROOT + "/outputs/single_xray_pipeline"
REPO_DIR = "/content/RadioVue"

In [ ]:
#@title 2. GPU, Drive and code
import os, sys, subprocess, zipfile, importlib.util
from pathlib import Path

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(smi.stdout[:700] or smi.stderr)
assert smi.returncode == 0, "No GPU. Runtime > Change runtime type > GPU (L4/A100 recommended, T4 minimum)."

from google.colab import drive
drive.mount("/content/drive")

bundle = Path("/content/drive/MyDrive") / CODE_BUNDLE_ON_DRIVE
if not Path(REPO_DIR, "radiovue_pipeline").exists():
    if bundle.is_file():
        Path(REPO_DIR).mkdir(parents=True, exist_ok=True)
        zipfile.ZipFile(bundle).extractall(REPO_DIR)
        print("Code from Drive bundle:", bundle)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
assert Path(REPO_DIR, "radiovue_pipeline").exists(), (
    "radiovue_pipeline/ not found. Push the branch that contains it, or upload the code bundle (SETUP_GUIDE.md).")
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
rev = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Code revision:", rev or "(bundle, see BUNDLE_MANIFEST.json)")

# colab_setup has no third-party imports, so it can run before packages are installed.
spec = importlib.util.spec_from_file_location("colab_setup", f"{REPO_DIR}/radiovue_pipeline/colab_setup.py")
cs = importlib.util.module_from_spec(spec); spec.loader.exec_module(cs)
GPU = cs.gpu_report()

In [ ]:
#@title 3. Install dependencies and stage models (first run ~10-15 min)
cs.install_runtime(parallax=True, dvg=False, tools=True)
cs.ensure_parallax_shims(f"{REPO_DIR}/Parallax")
PARALLAX_MODEL_DIR = cs.stage_parallax_model(PARALLAX_MODEL_ON_DRIVE)
HF_TOKEN = cs.hf_token_from_colab_secrets()  # value is never printed
assert HF_TOKEN, "Add HF_TOKEN in Colab Secrets (key icon) and enable notebook access."

In [ ]:
#@title Public CC0 test radiographs (same subject, same session; Wikimedia Commons, user Stillwaterising, 2010)
import hashlib, urllib.request
PUBLIC = {
    "pa_cc0_2010.png": ("https://upload.wikimedia.org/wikipedia/commons/c/c8/Chest_Xray_PA_3-8-2010.png",
                        "a43431b089a6dfab031c10eceef6c821e4f326913bf9015f73f4a1f0446d2be1"),
    "lateral_cc0_2010.png": ("https://upload.wikimedia.org/wikipedia/commons/9/9d/Chest_Xray_Lateral_3-8-2010.png",
                             "d07a306fb285a8456739e0349fb208b64ade12eb4bb84bc09889c7b3ec7f99a7"),
}
PUBLIC_DIR = Path("/content/public_cc0"); PUBLIC_DIR.mkdir(exist_ok=True)
for name, (url, digest) in PUBLIC.items():
    path = PUBLIC_DIR / name
    if not path.exists():
        req = urllib.request.Request(url, headers={"User-Agent": "RadioVueResearch/0.1"})
        path.write_bytes(urllib.request.urlopen(req).read())
    assert hashlib.sha256(path.read_bytes()).hexdigest() == digest, f"Checksum mismatch: {name}"
print("Public test images verified:", sorted(p.name for p in PUBLIC_DIR.iterdir()))

In [ ]:
#@title 4. Embeddings
import json, numpy as np, torch
from PIL import Image
from radiovue_pipeline.io_utils import load_xray
from radiovue_pipeline.embeddings import MedSigLIPEmbedder, ParallaxClipEmbedder, XrvDenseNetEmbedder, summarize
cs.check_gated_access("google/medsiglip-448")
pa = load_xray(PUBLIC_DIR / "pa_cc0_2010.png").image
lat = load_xray(PUBLIC_DIR / "lateral_cc0_2010.png").image
probes = {"pa": pa, "lateral": lat, "pa_mirrored": pa.transpose(Image.Transpose.FLIP_LEFT_RIGHT),
          "pa_upside_down": pa.transpose(Image.Transpose.FLIP_TOP_BOTTOM),
          "noise": Image.fromarray((np.random.default_rng(0).random((512, 512)) * 255).astype(np.uint8))}
report = {}
for label, factory in [("medsiglip-448", lambda: MedSigLIPEmbedder()),
                       ("parallax-clip", lambda: ParallaxClipEmbedder(PARALLAX_MODEL_DIR)),
                       ("xrv-densenet121", lambda: XrvDenseNetEmbedder(device="cuda"))]:
    model = factory()
    report[label] = summarize(label, {k: model.embed(v) for k, v in probes.items()})
    if label == "medsiglip-448":
        report[label]["zero_shot_view"] = {k: model.zero_shot(probes[k]) for k in ("pa", "lateral")}
    del model; torch.cuda.empty_cache()
for label, r in report.items():
    c = r["cosine"]
    print(f"{label:16s} dim={r['dim']:5d}  PA~mirror={c['pa']['pa_mirrored']:.3f}  PA~upside={c['pa']['pa_upside_down']:.3f}  "
          f"PA~lateral={c['pa']['lateral']:.3f}  PA~noise={c['pa']['noise']:.3f}")
print("MedSigLIP zero-shot:", json.dumps(report["medsiglip-448"]["zero_shot_view"], indent=1))
OUT = Path(OUTPUT_ROOT) / "embeddings"; OUT.mkdir(parents=True, exist_ok=True)
(OUT / "embedding_probe_report.json").write_text(json.dumps(report, indent=2))

**Interpreting the results.** Record them in `EXPERIMENT_RESULTS.md`. Parallax's `cc_projection` and its DiT cross-attention were
trained on CLIP features. Replacing CLIP with MedSigLIP therefore means retraining, even if MedSigLIP probes better.
The retraining data requirement is in `VIEW_SYNTHESIS_RESEARCH.md` §6.